In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('dataset/Netflix Content Strategy Genre & Rating Analysis/netflix_titles.csv')

df.fillna('Unknown', inplace=True)

df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

df.info()


In [ ]:
content_counts = df['type'].value_counts()
print(content_counts)

In [ ]:
top_genres = df['listed_in'].str.split(', ').explode().value_counts()
top_genres.head(10)

In [ ]:
top_countries = df['country'].str.split(', ').explode().value_counts()
print(top_countries.head(10))

In [ ]:
df_exploded = df.copy()
df_exploded['genre'] = df['listed_in'].str.split(', ')
df_exploded = df_exploded.explode('genre')


result = df_exploded.groupby(['country', 'genre']).size().unstack(fill_value=0)


top_5_countries = top_countries.index[:5]
top_5_genres = top_genres.index[:5]

final_analysis = result.loc[top_5_countries, top_5_genres].sort_values



final_analysis

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


df_subset = df_exploded.groupby(['country', 'genre']).size().unstack(fill_value=0)


top_5_countries = df_subset.sum(axis=1).nlargest(5).index
top_5_genres = df_subset.sum(axis=0).nlargest(5).index
data_to_plot = df_subset.loc[top_5_countries, top_5_genres]

plt.figure(figsize=(10, 6))
sns.heatmap(data_to_plot, annot=True, fmt='d', cmap='YlGnBu')

plt.title('Heatmap: Opportunities by Country & Genre (IntelRain Strategy)')
plt.show()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
df["description"] = df['description'].fillna('')

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['description'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


indices = pd.Series(df.index, index=df['title']).drop_duplicates()


def get_recommendations(title, cosine_sim=cosine_sim):
    if title not in indices:
        return "Title not found in the Netflix dataset."
    
    idx = indices[title]
        
    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:11]
    
    movie_indices = [i[0] for i in sim_scores]

    return df['title'].iloc[movie_indices]


print("'Stranger Things':")
print(get_recommendations('Stranger Things'))



In [ ]:
features = ['description', 'listed_in', 'director', 'cast']
for feature in features:
    df[feature] = df[feature].fillna('')

def create_soup(x):
    return x['description'] + ' ' + x['listed_in'] + ' ' + x['director'] + ' ' + x['cast']

df['combined_features'] = df.apply(create_soup, axis=1)


tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

cosine_sim_advanced = cosine_similarity(tfidf_matrix, tfidf_matrix)

def get_advanced_recommendations(title, cosine_sim=cosine_sim_advanced):
    if title not in indices:
        return "Title not found in the Netflix dataset."
    
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[movie_indices]

print(get_advanced_recommendations('Stranger Things'))
